# EX — Data Engineering & MLOps Real-World Exercises

An ETL pipeline, orchestration/DAG concepts, model versioning, effective visualization,
and advanced SQL with window functions (via built-in `sqlite3`).


## 1. A Simple ETL Pipeline
**Pointer:** design each step to be idempotent — safe to re-run without duplicating or corrupting data.

In [ ]:
import pandas as pd
import numpy as np

# EXTRACT (pretend this reads from an API/database)
def extract():
    return pd.DataFrame({
        "order_id": range(1, 21),
        "amount": np.random.uniform(10, 200, 20).round(2),
        "status": np.random.choice(["completed","completed","completed","refunded","pending"], 20),
        "customer_email": [f"user{i}@Example.COM" for i in range(1, 21)],
    })

raw = extract()
raw.head()


### TODO 1
Write `transform(df)` that: lowercases `customer_email`, drops `pending` orders, adds a `net_revenue` column (0 for refunded, else `amount`), then write `load(df, path)` that saves to CSV idempotently (overwrite, not append).

In [ ]:
# TODO
def transform(df):
    pass

def load(df, path):
    pass

clean = transform(raw)
load(clean, "/tmp/etl_output.csv")
print(clean.head())


<details><summary>Solution</summary>

```python
def transform(df):
    df = df.copy()
    df['customer_email'] = df['customer_email'].str.lower()
    df = df[df['status'] != 'pending']
    df['net_revenue'] = np.where(df['status']=='refunded', 0, df['amount'])
    return df

def load(df, path):
    df.to_csv(path, index=False)  # overwrite each run -- idempotent
```
</details>


## 2. Orchestration — Modeling Task Dependencies as a DAG
**Pointer:** this is exactly what tools like Airflow/Prefect do under the hood — model tasks as a graph, then run them in dependency order.

In [ ]:
# A tiny DAG executor: each task depends on others completing first
tasks = {
    "extract": {"depends_on": [], "fn": lambda: print("extracting...")},
    "transform": {"depends_on": ["extract"], "fn": lambda: print("transforming...")},
    "load_warehouse": {"depends_on": ["transform"], "fn": lambda: print("loading to warehouse...")},
    "load_reporting_db": {"depends_on": ["transform"], "fn": lambda: print("loading to reporting db...")},
    "notify": {"depends_on": ["load_warehouse", "load_reporting_db"], "fn": lambda: print("notifying...")},
}


### TODO 2
Write `run_dag(tasks)` that executes tasks in valid dependency order (topological sort) — a task only runs once all its `depends_on` tasks have run.

In [ ]:
# TODO
def run_dag(tasks):
    pass

run_dag(tasks)


<details><summary>Solution</summary>

```python
def run_dag(tasks):
    done = set()
    remaining = dict(tasks)
    while remaining:
        ready = [name for name, t in remaining.items() if all(d in done for d in t['depends_on'])]
        if not ready:
            raise ValueError("Cycle detected or unresolvable dependency!")
        for name in ready:
            remaining[name]['fn']()
            done.add(name)
            del remaining[name]
```
</details>


## 3. Model Versioning / Registry Concept

In [ ]:
import json, hashlib, datetime

model_registry = []

def register_model(name, params, metrics):
    version_hash = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:8]
    entry = {
        "name": name, "version": version_hash, "params": params,
        "metrics": metrics, "registered_at": datetime.datetime.now().isoformat(),
    }
    model_registry.append(entry)
    return entry

register_model("churn_classifier", {"n_estimators": 100, "max_depth": 4}, {"accuracy": 0.87})
register_model("churn_classifier", {"n_estimators": 200, "max_depth": 6}, {"accuracy": 0.89})
for e in model_registry:
    print(e["name"], e["version"], e["metrics"])


### TODO 3
Write `best_version(registry, name, metric)` returning the registry entry with the highest value of `metric` for a given model `name`.

In [ ]:
# TODO
def best_version(registry, name, metric):
    pass

print(best_version(model_registry, "churn_classifier", "accuracy"))


<details><summary>Solution</summary>

```python
def best_version(registry, name, metric):
    candidates = [e for e in registry if e['name']==name]
    return max(candidates, key=lambda e: e['metrics'][metric])
```
</details>

## 4. Effective Data Visualization
**Pointer:** a truncated y-axis can visually exaggerate a small difference — always show the honest scale unless you have a clear, labeled reason not to.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

values = [102, 105, 103, 108]
labels = ["Q1","Q2","Q3","Q4"]

fig, axes = plt.subplots(1, 2, figsize=(9,4))
axes[0].bar(labels, values); axes[0].set_ylim(0, 120); axes[0].set_title("Honest scale (starts at 0)")
axes[1].bar(labels, values); axes[1].set_ylim(100, 110); axes[1].set_title("Misleading scale (truncated axis)")
plt.tight_layout()
plt.savefig("/tmp/viz_comparison.png", dpi=80)
plt.close()
print("saved comparison plot -- notice how the right chart exaggerates a ~6% change to look dramatic")


## 5. Advanced SQL — Window Functions
**Pointer:** window functions solve 'running total' and 'rank within group' problems that plain GROUP BY can't.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute('''
CREATE TABLE sales (region TEXT, month TEXT, revenue INTEGER)
''')
rows = [
    ("North","Jan",100), ("North","Feb",150), ("North","Mar",120),
    ("South","Jan",80), ("South","Feb",90), ("South","Mar",130),
]
conn.executemany("INSERT INTO sales VALUES (?,?,?)", rows)
conn.commit()

query = '''
SELECT region, month, revenue,
       SUM(revenue) OVER (PARTITION BY region ORDER BY month) AS running_total,
       RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS revenue_rank
FROM sales
ORDER BY region, month
'''
for row in conn.execute(query):
    print(row)


### TODO 4
Write a SQL query finding, for each region, the month with the **highest** revenue (hint: filter where `revenue_rank = 1` using a subquery or CTE).

In [ ]:
# TODO
query2 = '''
-- your SQL here
'''
for row in conn.execute(query2):
    print(row)


<details><summary>Solution</summary>

```python
query2 = '''
WITH ranked AS (
  SELECT region, month, revenue,
         RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS rnk
  FROM sales
)
SELECT region, month, revenue FROM ranked WHERE rnk = 1
'''
for row in conn.execute(query2):
    print(row)
```
</details>


## Key Takeaways
- Design ETL steps to be idempotent — safe to re-run without side effects.
- Orchestration tools automate exactly what run_dag() does here: executing tasks in valid dependency order.
- Version models with their parameters and metrics together — reproducibility depends on it.
- Be deliberate and honest with axis scales; a misleading chart is a real risk in reporting.
- Window functions (`OVER (PARTITION BY ... ORDER BY ...)`) solve ranking/running-total problems GROUP BY can't.
